# Vision Transformer (ViT) Experimentation

This notebook is a sandbox for interactively testing and experimenting with the `CochleogramViT` model.

You can:
- Instantiate the model with different parameters.
- Pass dummy data to verify input/output shapes.
- Inspect model layers and parameter counts.


In [ ]:
from cochleogram_vit.models.vit import CochleogramViT
import torch

# 1. Instantiate the model with default parameters
# These defaults match the paper's architecture.
# UPDATE: Set channels to 3 to accept Viridis RGB images.
vit_model = CochleogramViT(
        image_size=128, patch_size=16, num_classes=4, dim=512,
        depth=6, heads=8, mlp_dim=1024, channels=3
        
    )

print("CochleogramViT model instantiated successfully for 3-channel input.")
# print(vit_model) # Uncomment to see the full architecture


## 2. Forward Pass Test

Create a dummy batch of tensors and pass it through the model to ensure the input and output dimensions are correct.


In [ ]:
# Create a dummy batch of 4 RGB cochleograms (Batch, Channels, Height, Width)
dummy_batch = torch.randn(4, 3, 128, 128) # Channels set to 3

# Perform a forward pass
with torch.no_grad():
    logits = vit_model(dummy_batch)

print(f"Input shape:  {dummy_batch.shape}")
print(f"Output shape: {logits.shape}")

# Check that the output shape is as expected (Batch, Num_Classes)
assert logits.shape == (4, 4)
print("\nSuccess! The model produced the correct output shape for 3-channel input.")


## 3. Load Configuration and Data

Now, let's load the dataset. We will use the `ICBHIDataset` class and the patient-wise split function from your `src` directory to prepare for training.


In [14]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import matplotlib.pyplot as plt
import torch

# Configuration
DATA_DIR = '../data/processed/cochleograms'
METADATA_PATH = '../data/processed/metadata.csv'
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.0001

# Custom Dataset
class CochleogramDataset(Dataset):
    def __init__(self, data_dir, metadata_path, transform=None):
        self.data_dir = data_dir
        self.metadata = pd.read_csv(metadata_path)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        npy_path = os.path.join(self.data_dir, os.path.basename(row['npy_path']))
        cochleogram = np.load(npy_path)  # already [0,1], no need to renormalize
        label = int(row['label'])

        # Apply Viridis colormap
        viridis_cmap = plt.get_cmap('viridis')
        colored_cochleogram = viridis_cmap(cochleogram)

        # Drop alpha, transpose to (C, H, W), make contiguous
        rgb_cochleogram = np.ascontiguousarray(colored_cochleogram[:, :, :3].transpose(2, 0, 1))
        cochleogram_tensor = torch.from_numpy(rgb_cochleogram).float()

        if self.transform:
            cochleogram_tensor = self.transform(cochleogram_tensor)

        return cochleogram_tensor, label

# Create dataset
dataset = CochleogramDataset(DATA_DIR, METADATA_PATH)

print(f"Dataset size: {len(dataset)}")
sample_img, sample_label = dataset[0]
print(f"Sample image shape: {sample_img.shape}")  # Should be (3, 128, 128)
print(f"Min: {sample_img.min():.4f}, Max: {sample_img.max():.4f}")
print(f"Any NaN: {torch.isnan(sample_img).any()}")
print(f"Any Inf: {torch.isinf(sample_img).any()}")

Dataset size: 6898
Sample image shape: torch.Size([3, 128, 128])
Min: 0.0049, Max: 0.8719
Any NaN: False
Any Inf: False


## 4. Train the Model

Now we'll set up the optimizer and loss function and run a basic training loop.


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm.notebook import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import numpy as np
import copy

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Reproducibility ---
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# --- Cross-Validation Setup ---
metadata = dataset.metadata
metadata['patient_id'] = metadata['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])
groups = metadata['patient_id'].values
gkf = GroupKFold(n_splits=10)

# --- Class Weights (softened with power 0.75) ---
raw_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2, 3]),
    y=metadata['label'].values
)
class_weights = raw_weights ** 0.75
class_weights = class_weights / class_weights.sum() * len(class_weights)  # renormalize

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nClass weights (softened ^0.75, renormalized):")
print(f"  Normal   (0): {class_weights[0]:.4f}")
print(f"  Crackles (1): {class_weights[1]:.4f}")
print(f"  Wheezes  (2): {class_weights[2]:.4f}")
print(f"  Both     (3): {class_weights[3]:.4f}")

# --- LR Warmup + Cosine Decay ---
def lr_lambda(epoch):
    warmup_epochs = 4
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    denom = EPOCHS - warmup_epochs
    if denom == 0:
        return 0.0
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / denom))

# Store results
fold_results = []
all_preds_total = []
all_labels_total = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(metadata, groups=groups)):
    print('\n' + '='*60)
    print(f'FOLD {fold+1}/10')
    print('='*60)

    # --- WeightedRandomSampler for training ---
    train_labels = metadata['label'].values[train_idx]
    sample_weights = torch.tensor(
        [class_weights[l] for l in train_labels],
        dtype=torch.float
    )
    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    # --- SubsetRandomSampler for validation (no balancing) ---
    val_sampler = torch.utils.data.SubsetRandomSampler(val_idx)

    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=train_sampler)
    val_loader   = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=val_sampler)

    print(f"  Train samples: {len(train_idx)} | Val samples: {len(val_idx)}")
    print(f"  Train class distribution: {dict(sorted(Counter(train_labels.tolist()).items()))}")

    # --- Fixed seed per fold for reproducibility ---
    torch.manual_seed(42 + fold)
    np.random.seed(42 + fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42 + fold)

    # --- Re-initialize model and optimizer for each fold ---
    vit_model = CochleogramViT(
        image_size=128, patch_size=16, num_classes=4, dim=512,
        depth=6, heads=8, mlp_dim=1024, channels=3,
        dropout=0.3, emb_dropout=0.2
    ).to(device)

    optimizer = optim.Adam(vit_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # --- Training Loop ---
    print(f"\n  {'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14} {'LR':<12} {'Status'}")
    print(f"  {'-'*60}")

    best_score = 0.0
    best_model_state = None
    best_epoch = 1

    for epoch in range(EPOCHS):
        # Training phase
        vit_model.train()
        running_loss = 0.0
        for cochleograms, labels in tqdm(train_loader, desc=f"  Epoch {epoch+1}/{EPOCHS}", leave=False):
            cochleograms, labels = cochleograms.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = vit_model(cochleograms)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)

        # Validation phase
        vit_model.eval()
        val_loss = 0.0
        val_preds = []
        val_labels_epoch = []
        with torch.no_grad():
            for cochleograms, labels in val_loader:
                cochleograms, labels = cochleograms.to(device), labels.to(device)
                outputs = vit_model(cochleograms)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_epoch.extend(labels.cpu().numpy())
        val_loss /= len(val_loader)

        # ── Per-epoch metric block (used for best checkpoint selection) ──
        val_preds_arr  = np.array(val_preds)
        val_labels_arr = np.array(val_labels_epoch)

        TP_e = np.sum((val_labels_arr != 0) & (val_preds_arr == val_labels_arr))
        FN_e = np.sum((val_labels_arr != 0) & (val_preds_arr == 0))                                                     # paper def: adventitious → normal
        FN_wrong_type_e = np.sum((val_labels_arr != 0) & (val_preds_arr != 0) & (val_preds_arr != val_labels_arr))     # subtype confusion, stored only
        TN_e = np.sum((val_labels_arr == 0) & (val_preds_arr == 0))
        FP_e = np.sum((val_labels_arr == 0) & (val_preds_arr != 0))

        assert FN_e + FN_wrong_type_e + TP_e == np.sum(val_labels_arr != 0), "Epoch adventitious decomposition mismatch"

        sensitivity_e = TP_e / (TP_e + FN_e + 1e-8)
        specificity_e = TN_e / (TN_e + FP_e + 1e-8)
        epoch_score   = (sensitivity_e + specificity_e) / 2.0

        # Save best checkpoint based on score
        if epoch_score > best_score:
            best_score = epoch_score
            best_model_state = copy.deepcopy(vit_model.state_dict())
            best_epoch = epoch + 1

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

    print(f"\n  Best checkpoint at epoch {best_epoch} with Score: {best_score*100:.2f}%")

    # --- Load best model for evaluation ---
    vit_model.load_state_dict(best_model_state)

    # --- Evaluation ---
    print(f"\n  Evaluating fold {fold+1}...")
    vit_model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for cochleograms, labels in val_loader:
            cochleograms, labels = cochleograms.to(device), labels.to(device)
            outputs = vit_model(cochleograms)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Accumulate for aggregated CM
    all_preds_total.extend(all_preds)
    all_labels_total.extend(all_labels)

    print(f"  True label distribution:      {dict(sorted(Counter(all_labels).items()))}")
    print(f"  Predicted label distribution: {dict(sorted(Counter(all_preds).items()))}")

    # ── Per-fold metric block ──
    all_preds_arr  = np.array(all_preds)
    all_labels_arr = np.array(all_labels)

    TP = np.sum((all_labels_arr != 0) & (all_preds_arr == all_labels_arr))
    FN = np.sum((all_labels_arr != 0) & (all_preds_arr == 0))                                                      # paper def: adventitious → normal
    FN_wrong_type = np.sum((all_labels_arr != 0) & (all_preds_arr != 0) & (all_preds_arr != all_labels_arr))      # subtype confusion, stored only
    TN = np.sum((all_labels_arr == 0) & (all_preds_arr == 0))
    FP = np.sum((all_labels_arr == 0) & (all_preds_arr != 0))

    assert FN + FN_wrong_type + TP == np.sum(all_labels_arr != 0), "Fold adventitious decomposition mismatch"

    sensitivity = TP / (TP + FN + 1e-8)
    specificity = TN / (TN + FP + 1e-8)
    precision   = TP / (TP + FP + 1e-8)
    accuracy    = (TP + TN) / (TP + TN + FP + FN + 1e-8)
    score       = (sensitivity + specificity) / 2.0

    fold_results.append({
        'fold': fold + 1,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'accuracy': accuracy,
        'score': score,
        # stored for later analysis
        'FN_wrong_type': FN_wrong_type,
    })

    print(f"\n  --- Fold {fold+1} Results ---")
    print(f"  Accuracy:    {accuracy*100:.2f}%")
    print(f"  Sensitivity: {sensitivity*100:.2f}%")
    print(f"  Specificity: {specificity*100:.2f}%")
    print(f"  Precision:   {precision*100:.2f}%")
    print(f"  Score:       {score*100:.2f}%")
    print(f"  TP={TP}  FN={FN}  TN={TN}  FP={FP}  FN_wrong_type={FN_wrong_type}")


# ── Aggregated metrics across ALL folds ──────────────────────────────────────
print("\n" + "="*60)
print("AGGREGATED 10-FOLD RESULTS")
print("="*60)

all_preds_arr  = np.array(all_preds_total)
all_labels_arr = np.array(all_labels_total)

TP = np.sum((all_labels_arr != 0) & (all_preds_arr == all_labels_arr))
FN = np.sum((all_labels_arr != 0) & (all_preds_arr == 0))                                                      # paper def: adventitious → normal
FN_wrong_type = np.sum((all_labels_arr != 0) & (all_preds_arr != 0) & (all_preds_arr != all_labels_arr))      # subtype confusion, stored only
TN = np.sum((all_labels_arr == 0) & (all_preds_arr == 0))
FP = np.sum((all_labels_arr == 0) & (all_preds_arr != 0))

assert FN + FN_wrong_type + TP == np.sum(all_labels_arr != 0), "Aggregated adventitious decomposition mismatch"

sensitivity = TP / (TP + FN + 1e-8)
specificity = TN / (TN + FP + 1e-8)
precision   = TP / (TP + FP + 1e-8)
accuracy    = (TP + TN) / (TP + TN + FP + FN + 1e-8)
score       = (sensitivity + specificity) / 2.0

print(f"  Accuracy:    {accuracy*100:.2f}%")
print(f"  Sensitivity: {sensitivity*100:.2f}%")
print(f"  Specificity: {specificity*100:.2f}%")
print(f"  Precision:   {precision*100:.2f}%")
print(f"  Score:       {score*100:.2f}%")
print(f"  TP={TP}  FN={FN}  TN={TN}  FP={FP}  FN_wrong_type={FN_wrong_type}")

# ── Per-class metrics (one-vs-rest) ──────────────────────────────────────────
# Note: this section uses the 4-class confusion matrix directly so no changes needed here
print("\n" + "="*60)
print("PER-CLASS RESULTS (One-vs-Rest)")
print("="*60)

class_names = ['Normal', 'Crackles', 'Wheezes', 'Both']
agg_cm_4class = confusion_matrix(all_labels_total, all_preds_total, labels=list(range(4)))
print("\n  4-Class Confusion Matrix:")
print(f"  {'':12}", end="")
for name in class_names:
    print(f"  {name:<10}", end="")
print()
for i, name in enumerate(class_names):
    print(f"  {name:<12}", end="")
    for j in range(4):
        print(f"  {agg_cm_4class[i,j]:<10}", end="")
    print()

for c in range(4):
    TP_c = agg_cm_4class[c, c]
    FN_c = agg_cm_4class[c, :].sum() - TP_c
    FP_c = agg_cm_4class[:, c].sum() - TP_c
    TN_c = agg_cm_4class.sum() - TP_c - FN_c - FP_c

    sen_c = TP_c / (TP_c + FN_c + 1e-8)
    spe_c = TN_c / (TN_c + FP_c + 1e-8)
    pre_c = TP_c / (TP_c + FP_c + 1e-8)
    acc_c = (TP_c + TN_c) / (agg_cm_4class.sum() + 1e-8)
    sco_c = (sen_c + spe_c) / 2.0

    print(f"\n  [{class_names[c]}]")
    print(f"    Sensitivity: {sen_c*100:.2f}%")
    print(f"    Specificity: {spe_c*100:.2f}%")
    print(f"    Precision:   {pre_c*100:.2f}%")
    print(f"    Accuracy:    {acc_c*100:.2f}%")
    print(f"    Score:       {sco_c*100:.2f}%")

print("\n" + "="*60)
print("Cross-validation training finished.")
print("="*60)

In [ ]:
# ── Strict Sensitivity per paper definition ───────────────────────────────────

print("\n" + "="*60)
print("STRICT SENSITIVITY (TP = correct adventitious class)")
print("="*60)

all_preds_arr  = np.array(all_preds_total)
all_labels_arr = np.array(all_labels_total)

# Reconstruct fold boundaries
fold_sizes = []
for _, val_idx in gkf.split(metadata, groups=groups):
    fold_sizes.append(len(val_idx))

print(f"\n  {'Fold':<6} {'Sensitivity':>13} {'Specificity':>13} {'Score':>10}")
print(f"  {'-'*46}")

cursor = 0
fold_scores = []
for fold_idx, size in enumerate(fold_sizes):
    fold_preds  = all_preds_arr[cursor:cursor + size]
    fold_labels = all_labels_arr[cursor:cursor + size]
    cursor += size

    # TP: adventitious correctly classified (exact match)
    TP = np.sum((fold_labels != 0) & (fold_preds == fold_labels))
    # FN: adventitious predicted as anything other than correct class
    FN = np.sum((fold_labels != 0) & (fold_preds != fold_labels))
    # TN: normal correctly classified as Normal
    TN = np.sum((fold_labels == 0) & (fold_preds == 0))
    # FP: normal incorrectly classified as adventitious
    FP = np.sum((fold_labels == 0) & (fold_preds != 0))

    sensitivity = TP / (TP + FN + 1e-8)
    specificity = TN / (TN + FP + 1e-8)
    score       = (sensitivity + specificity) / 2.0
    fold_scores.append(score)

    print(f"  Fold {fold_idx+1:<2}"
          f"  {sensitivity*100:>11.2f}%"
          f"  {specificity*100:>11.2f}%"
          f"  {score*100:>8.2f}%")

# Aggregated
print(f"\n  {'-'*46}")
TP = np.sum((all_labels_arr != 0) & (all_preds_arr == all_labels_arr))
FN = np.sum((all_labels_arr != 0) & (all_preds_arr != all_labels_arr))
TN = np.sum((all_labels_arr == 0) & (all_preds_arr == 0))
FP = np.sum((all_labels_arr == 0) & (all_preds_arr != 0))

sensitivity = TP / (TP + FN + 1e-8)
specificity = TN / (TN + FP + 1e-8)
precision   = TP / (TP + FP + 1e-8)
accuracy    = (TP + TN) / (TP + TN + FP + FN + 1e-8)
score       = (sensitivity + specificity) / 2.0

print(f"  {'AGG':<6}"
      f"  {sensitivity*100:>11.2f}%"
      f"  {specificity*100:>11.2f}%"
      f"  {score*100:>8.2f}%")

print(f"\n  Accuracy:    {accuracy*100:.2f}%")
print(f"  Precision:   {precision*100:.2f}%")
print(f"  TP={TP}  FN={FN}  TN={TN}  FP={FP}")
print("="*60)